In [37]:
import os
os.makedirs("family_tree", exist_ok=True)

In [60]:
%%writefile family_tree/person.py
class Person:
    def __init__(self, name, surname, gender, age, middle_name=None):
        self.name = name
        self.surname = surname
        self.gender = gender
        self.age = age
        self.middle_name = middle_name
        self.father = None
        self.mother = None
        self.partner = None

    def set_middle_from_father(self, father):
        if father is None:
            return
        if self.gender == 'm':
            self.middle_name = father.name + "ович"
        else:
            self.middle_name = father.name + "овна"

    def full_name(self):
        if self.middle_name:
            return f"{self.surname} {self.name} {self.middle_name}"
        return f"{self.surname} {self.name}"

    def __str__(self):
        return f"{self.full_name()} ({self.age})"


class Male(Person):
    def __init__(self, name, surname, age, middle_name=None):
        super().__init__(name, surname, 'm', age, middle_name)


class Female(Person):
    def __init__(self, name, surname, age, middle_name=None):
        super().__init__(name, surname, 'f', age, middle_name)

Overwriting family_tree/person.py


In [74]:
%%writefile family_tree/family.py
from person import Male, Female

TAB = 3


class Family:
    def __init__(self, surname, description=""):
        self.surname = surname
        self.description = description
        self.members = []

    def family_name(self):
        s = self.surname
        if s.endswith(("ов", "ев", "ин", "ын")):
            return s + "а"
        return s

    def add_member(self, person):
        self.members.append(person)

    def add_partner(self, p1, p2):
        p1.partner = p2
        p2.partner = p1

    def add_child(self, name, gender, age, father=None, mother=None, middle_name=None):
        if gender == 'm':
            child = Male(name, self.surname, age)
        else:
            child = Female(name, self.surname, age)

        if middle_name is not None:
            child.middle_name = middle_name
        elif father is not None:
            child.set_middle_from_father(father)

        child.father = father
        child.mother = mother
        self.members.append(child)
        return child

    def _generation(self, person):
        """0 — дедушки/бабушки, 1 — их дети, 2 — внуки, 3 — правнуки."""
        gen = 0
        cur = person
        while cur.father is not None or cur.mother is not None:
            cur = cur.father if cur.father is not None else cur.mother
            gen += 1
        return gen

    def _tab(self, person):
        return " " * (TAB * self._generation(person))

    def _children_of(self, person):
        return [c for c in self.members
                if c.father is person or c.mother is person]

    def print_tree(self):
        print(f"семья {self.family_name()}:")
        if self.description:
            print(f"  ({self.description})")

        roots = [p for p in self.members
                 if p.father is None and p.mother is None]

        printed = set()
        for root in roots:
            self._print_branch(root, printed)

    def _print_branch(self, person, printed):
        if person in printed:
            return

        print(f"{self._tab(person)}{person}")
        printed.add(person)

        if person.partner and person.partner not in printed:
            print(f"{self._tab(person)}{person.partner}")
            printed.add(person.partner)

        for child in self._children_of(person):
            self._print_branch(child, printed)


class NewBranchFamily(Family):
    """Особая ветвь: отчество задаётся вручную"""

    def add_child(self, name, gender, age, mother=None, middle_name=None):
        if gender == 'm':
            child = Male(name, self.surname, age, middle_name)
        else:
            child = Female(name, self.surname, age, middle_name)
        child.mother = mother
        self.members.append(child)
        return child

Overwriting family_tree/family.py


In [71]:
%%writefile family_tree/city.py
class City:
    def __init__(self, name):
        self.name = name
        self.families = []

    def add_family(self, family):
        self.families.append(family)

    def find_family(self, surname):
        for f in self.families:
            if f.surname == surname:
                return f
        return None

    def stats(self):
        total = sum(len(f.members) for f in self.families)
        print(f"город {self.name}: семей — {len(self.families)}, человек — {total}")

    def print_tree(self):
        """Печатает деревья всех семей города и общую статистику."""
        print(f"город {self.name}")
        for f in self.families:
            f.print_tree()
            print()
        self.stats()

Overwriting family_tree/city.py


In [77]:
%%writefile family_tree/main.py
from person import Male, Female
from family import Family, NewBranchFamily
from city import City


def main():
    city = City("Н")

    ivanovy = Family(
        "Иванов",
        "дедушка Иван и бабушка Ольга вырастили двух сыновей — Петра и Сергея. "
        "У Петра двое детей, у Сергея двое детей."
    )

    ded = Male("Иван", "Иванов", 70, "Петрович")
    baba = Female("Ольга", "Иванова", 68, "Сергеевна")
    ivanovy.add_member(ded)
    ivanovy.add_member(baba)
    ivanovy.add_partner(ded, baba)

    petr = ivanovy.add_child("Пётр", 'm', 45, father=ded)
    sergey = ivanovy.add_child("Сергей", 'm', 42, father=ded)

    maria = Female("Мария", "Иванова", 43, "Сергеевна")
    anna = Female("Анна", "Иванова", 40, "Ивановна")
    ivanovy.add_member(maria)
    ivanovy.add_member(anna)
    ivanovy.add_partner(petr, maria)
    ivanovy.add_partner(sergey, anna)

    ivanovy.add_child("Пётр", 'm', 15, father=petr, mother=maria)
    ivanovy.add_child("Анна", 'f', 10, father=petr, mother=maria)
    ivanovy.add_child("Олег", 'm', 12, father=sergey, mother=anna)
    ivanovy.add_child("Мария", 'f', 8, father=sergey, mother=anna)

    petrovy = Family(
        "Петров",
        "воспитывают родного сына Дмитрия и приёмного Олега. "
        "У Олега отчество задано вручную"
    )

    sergey_p = Male("Сергей", "Петров", 50, "Николаевич")
    natasha = Female("Наталья", "Петрова", 48, "Ивановна")
    petrovy.add_member(sergey_p)
    petrovy.add_member(natasha)
    petrovy.add_partner(sergey_p, natasha)

    petrovy.add_child("Дмитрий", 'm', 20, father=sergey_p, mother=natasha)
    petrovy.add_child("Олег", 'm', 12, father=sergey_p, mother=natasha,
                      middle_name="Иванович")

    goncharovy = NewBranchFamily(
        "Гончаров",
        "Татьяна одна воспитывает сына Андрея. Отца в семье нет, "
        "отчество задано вручную."
    )

    tatyana = Female("Татьяна", "Гончарова", 55, "Петровна")
    goncharovy.add_member(tatyana)
    goncharovy.add_child("Андрей", 'm', 25, mother=tatyana,
                         middle_name="Викторович")

    city.add_family(ivanovy)
    city.add_family(petrovy)
    city.add_family(goncharovy)

    city.print_tree()


if __name__ == "__main__":
    main()

Overwriting family_tree/main.py


In [78]:
!python family_tree/main.py

город Н
семья Иванова:
  (дедушка Иван и бабушка Ольга вырастили двух сыновей — Петра и Сергея. У Петра двое детей, у Сергея двое детей.)
Иванов Иван Петрович (70)
Иванова Ольга Сергеевна (68)
   Иванов Пётр Иванович (45)
   Иванова Мария Сергеевна (43)
      Иванов Пётр Пётрович (15)
      Иванов Анна Пётровна (10)
   Иванов Сергей Иванович (42)
   Иванова Анна Ивановна (40)
      Иванов Олег Сергейович (12)
      Иванов Мария Сергейовна (8)

семья Петрова:
  (воспитывают родного сына Дмитрия и приёмного Олега. У Олега отчество задано вручную)
Петров Сергей Николаевич (50)
Петрова Наталья Ивановна (48)
   Петров Дмитрий Сергейович (20)
   Петров Олег Иванович (12)

семья Гончарова:
  (Татьяна одна воспитывает сына Андрея. Отца в семье нет, отчество задано вручную.)
Гончарова Татьяна Петровна (55)
   Гончаров Андрей Викторович (25)

город Н: семей — 3, человек — 16
